In [1]:
from langgraph.graph import StateGraph,START, END
from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage,BaseMessage
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_core.tools import  tool
from langchain_google_vertexai import ChatVertexAI

import requests
import urllib3
import ssl
import os

c:\Users\H604660\Desktop\Siddhi Shintre\My Learnings\LANGGRAPH\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\H604660\Desktop\Siddhi Shintre\My Learnings\LANGGRAPH\venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.7) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\H604660\Desktop\Siddhi Shintre\My Learnings\LANGGRAPH\venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.7) which Google will stop supporting in new

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter  
from langchain_community.vectorstores import FAISS



In [5]:
llm = ChatVertexAI(
    model_name="gemini-2.0-flash",
    project=os.getenv('GEMINI_API_KEY'),
    location="us-central1",
    temperature=0.3,  # Lower temperature for more focused responses
    max_output_tokens=50,  # Limit response length (adjust as needed: 50-200)
)

# # Step 3: Bind tools to the LLM
# llm_with_tools = llm.bind_tools(tools)

C:\Users\H604660\AppData\Local\Temp\ipykernel_704\575448132.py:1: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  llm = ChatVertexAI(
C:\Users\H604660\AppData\Local\Temp\ipykernel_704\575448132.py:1: LangChainDeprecationWarning: The class `ChatVertexAI` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import ChatGoogleGenerativeAI``.
  llm = ChatVertexAI(


In [8]:
loader = PyPDFLoader("ML_Book.pdf")
docs = loader.load()

In [15]:
print(docs)
print(len(docs))

print(type(docs))

print(docs[0])
print("-"*50)
print(docs[0].page_content)

print("-"*50)
print(docs[0].metadata)

[Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2016-09-21T13:04:39+00:00', 'author': 'Andreas C. Müller and Sarah Guido', 'title': 'Introduction to Machine Learning with Python', 'trapped': '/False', 'moddate': '2020-08-19T07:09:16+02:00', 'source': 'ML_Book.pdf', 'total_pages': 392, 'page': 0, 'page_label': 'Cover'}, page_content='Andreas C. Müller & Sarah Guido\nIntroduction to \nMachine \nLearning  \nwith P y t h o n   \nA GUIDE FOR DATA SCIENTISTS'), Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2016-09-21T13:04:39+00:00', 'author': 'Andreas C. Müller and Sarah Guido', 'title': 'Introduction to Machine Learning with Python', 'trapped': '/False', 'moddate':

In [17]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 973


In [20]:
bge_em = "C:\\Users\\H604660\\Downloads\\bgem3weights"
#from BGE documentation
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name=bge_em, 
encode_kwargs={"normalize_embeddings": True})

vector_store = FAISS.from_documents(chunks, embedding_model)






Loading weights: 100%|██████████| 391/391 [00:00<00:00, 12066.17it/s]


In [21]:
# (optional) save the vector database to a local directory
vector_store.save_local("vectorstore.db")

In [22]:
retriver = vector_store.as_retriever(search_type='similarity', search_kwargs={"k": 5})

In [24]:
retriver.invoke("What is machine learning?")

[Document(id='2f2e1367-1090-4ded-9223-7a1e640b6efa', metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2016-09-21T13:04:39+00:00', 'author': 'Andreas C. Müller and Sarah Guido', 'title': 'Introduction to Machine Learning with Python', 'trapped': '/False', 'moddate': '2020-08-19T07:09:16+02:00', 'source': 'ML_Book.pdf', 'total_pages': 392, 'page': 14, 'page_label': '1'}, page_content='CHAPTER 1\nIntroduction\nMachine learning is about extracting knowledge from data. It is a research field at the\nintersection of statistics, artificial intelligence, and computer science and is also\nknown as predictive analytics or statistical learning. The application of machine\nlearning methods has in recent years become ubiquitous in everyday life. From auto‐\nmatic recommendations of which movies to watch, to what food to order or which\nproducts to

In [23]:
#HEre result = retriver.invoke(query) Query is automatically get embedded.
#The query only gets embedded when we are using a Vector Store Retriever.
# If we were using a Keyword Retriever (like BM25Retriever), it would not embed the query. Instead, it would just look for matching words (like "capital" and "France") using traditional search math (TF-IDF), without using vectors at all.


@tool
def rag_tool(query):
    """Use this tool to answer questions based on the content of the PDF document."""
    
    result = retriver.invoke(query)

    context = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]

    return {
        'query':query,
        'context':context,
        'metadata':metadata
    }

In [25]:
tools = [rag_tool]
llm_with_tools = llm.bind_tools(tools)

In [26]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [45]:
def chat_node(state: ChatState):
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {'messages':[response]}

In [46]:
tool_node= ToolNode(tools)

In [47]:
graph = StateGraph(ChatState)

graph.add_node('chat_node',chat_node)
graph.add_node('tools',tool_node)


graph.add_edge(START,'chat_node')
graph.add_conditional_edges('chat_node',tools_condition)
graph.add_edge('tools','chat_node')

chatbot = graph.compile()

In [48]:
#Displaying the graph
print(chatbot.get_graph().draw_ascii())

        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
        +-----------+         
        | chat_node |         
        +-----------+         
          .         .         
        ..           ..       
       .               .      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


In [49]:
result = chatbot.invoke(
    {
        "messages":[
            HumanMessage(
                content=(
                "What is machine learning?"
                )
                )
        ]
    }
)



c:\Users\H604660\Desktop\Siddhi Shintre\My Learnings\LANGGRAPH\venv\lib\site-packages\google\auth\_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [54]:
print(result)
print(type(result))
print(type(result['messages'][0]))
print(result['messages'][-1].content)

{'messages': [HumanMessage(content='What is machine learning?', additional_kwargs={}, response_metadata={}, id='9fcd0258-6c6e-41b0-bcee-d5204b4d9163'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'rag_tool', 'arguments': '{"query": "What is machine learning?"}'}}, response_metadata={'is_blocked': False, 'safety_ratings': [], 'usage_metadata': {'prompt_token_count': 26, 'candidates_token_count': 9, 'total_token_count': 35, 'prompt_tokens_details': [{'modality': 1, 'token_count': 26}], 'candidates_tokens_details': [{'modality': 1, 'token_count': 9}], 'thoughts_token_count': 0, 'cached_content_token_count': 0, 'cache_tokens_details': []}, 'finish_reason': 'STOP', 'avg_logprobs': -1.6619619499478074e-05, 'model_provider': 'google_vertexai', 'model_name': 'gemini-2.0-flash'}, id='lc_run--019d7bca-7b65-7721-a2d3-ec63bd074fcd-0', tool_calls=[{'name': 'rag_tool', 'args': {'query': 'What is machine learning?'}, 'id': '01103006-5d75-405e-870d-a1f5bd31d7fd', 'type': 'tool_c